In [3]:
# ============================================================
# CHANGE CAPTIONING — BASELINE
# LEVIR-CC
# ============================================================

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights

from tqdm.auto import tqdm

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

CC_ROOT = Path(
    r"C:\College\Dissertation\Data\LEVIR-CC\Levir-CC-dataset-001"
)

CC_IMAGE_ROOT = CC_ROOT / "images"

CAPTION_JSON = CC_ROOT / "LevirCCcaptions.json"

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

IMAGE_SIZE = 224

BATCH_SIZE = 32

NUM_EPOCHS = 10

LEARNING_RATE = 1e-4

NUM_WORKERS = 0

MAX_CAPTION_LENGTH = 30

print("\nConfiguration loaded.")

PyTorch: 2.11.0+cu128
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU

Configuration loaded.


In [2]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [4]:
# ============================================================
# LOAD LEVIR-CC CAPTIONS
# ============================================================

with open(CAPTION_JSON, "r", encoding="utf-8") as f:
    caption_data = json.load(f)

records = []

for item in caption_data["images"]:

    filename = item["filename"]
    split = item["split"]

    image_path_A = (
        CC_IMAGE_ROOT
        / split
        / "A"
        / filename
    )

    image_path_B = (
        CC_IMAGE_ROOT
        / split
        / "B"
        / filename
    )

    # Keep only samples whose A and B actually exist
    if not image_path_A.exists():
        continue

    if not image_path_B.exists():
        continue

    captions = [
        s["raw"].strip().lower()
        for s in item["sentences"]
    ]

    records.append({
        "filename": filename,
        "split": split,
        "image_A": str(image_path_A),
        "image_B": str(image_path_B),
        "changeflag": item["changeflag"],
        "captions": captions
    })

dataset_df = pd.DataFrame(records)

print("Total usable caption records:", len(dataset_df))

print("\nBy split:")
print(dataset_df["split"].value_counts())

print("\nChange distribution:")
print(dataset_df["changeflag"].value_counts())

display(dataset_df.head())

Total usable caption records: 10077

By split:
split
train    6815
test     1929
val      1333
Name: count, dtype: int64

Change distribution:
changeflag
0    5039
1    5038
Name: count, dtype: int64


,filename,split,image_A,image_B,changeflag,captions
0,train_000001.png,train,C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...,C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...,0,"[there is no difference ., the two scenes seem..."
1,train_000002.png,train,C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...,C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...,0,"[there is no difference ., the two scenes seem..."
2,train_000003.png,train,C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...,C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...,0,"[there is no difference ., the two scenes seem..."
3,train_000004.png,train,C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...,C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...,0,"[there is no difference ., the two scenes seem..."
4,train_000005.png,train,C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...,C:\College\Dissertation\Data\LEVIR-CC\Levir-CC...,0,"[there is no difference ., the two scenes seem..."


In [5]:
# ============================================================
# BUILD VOCABULARY
# ============================================================

from collections import Counter

SPECIAL_TOKENS = [
    "<PAD>",
    "<START>",
    "<END>",
    "<UNK>"
]

counter = Counter()

for captions in dataset_df["captions"]:

    for caption in captions:

        tokens = caption.replace(".", " .").split()

        counter.update(tokens)

# Keep reasonably frequent words
MIN_WORD_FREQ = 2

vocab_words = [
    word
    for word, count in counter.items()
    if count >= MIN_WORD_FREQ
]

itos = SPECIAL_TOKENS + sorted(vocab_words)

stoi = {
    word: idx
    for idx, word in enumerate(itos)
}

PAD_IDX = stoi["<PAD>"]
START_IDX = stoi["<START>"]
END_IDX = stoi["<END>"]
UNK_IDX = stoi["<UNK>"]

VOCAB_SIZE = len(stoi)

print("Vocabulary size:", VOCAB_SIZE)
print("Most common words:")
print(counter.most_common(20))

Vocabulary size: 733
Most common words:
[('.', 50385), ('the', 47102), ('a', 14170), ('is', 14140), ('has', 12025), ('road', 11163), ('and', 11146), ('built', 11017), ('are', 10257), ('no', 10086), ('houses', 10072), ('on', 8998), ('of', 8727), ('two', 8146), ('scene', 7328), ('in', 6766), ('some', 6341), ('roads', 5853), ('appear', 5319), ('as', 5142)]


In [6]:
# ============================================================
# DATASET
# ============================================================

class LEVIRCCDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None,
        max_length=30
    ):

        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.max_length = max_length

    def tokenize(self, caption):

        tokens = (
            caption
            .replace(".", " .")
            .split()
        )

        tokens = (
            ["<START>"]
            + tokens
            + ["<END>"]
        )

        tokens = tokens[:self.max_length]

        ids = [
            stoi.get(token, UNK_IDX)
            for token in tokens
        ]

        # Pad
        ids += [
            PAD_IDX
        ] * (self.max_length - len(ids))

        return torch.tensor(
            ids,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_A = Image.open(
            row["image_A"]
        ).convert("RGB")

        image_B = Image.open(
            row["image_B"]
        ).convert("RGB")

        if self.transform:

            image_A = self.transform(image_A)
            image_B = self.transform(image_B)

        caption = random.choice(
            row["captions"]
        )

        caption_ids = self.tokenize(
            caption
        )

        return (
            image_A,
            image_B,
            caption_ids
        )


# ------------------------------------------------------------
# Image transforms
# ------------------------------------------------------------

weights = ResNet18_Weights.DEFAULT

transform = T.Compose([
    T.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    T.ToTensor(),
    T.Normalize(
        mean=weights.transforms().mean,
        std=weights.transforms().std
    )
])


# ------------------------------------------------------------
# Split data
# ------------------------------------------------------------

train_df = dataset_df[
    dataset_df["split"] == "train"
].copy()

val_df = dataset_df[
    dataset_df["split"] == "val"
].copy()

test_df = dataset_df[
    dataset_df["split"] == "test"
].copy()


train_dataset = LEVIRCCDataset(
    train_df,
    transform=transform,
    max_length=MAX_CAPTION_LENGTH
)

val_dataset = LEVIRCCDataset(
    val_df,
    transform=transform,
    max_length=MAX_CAPTION_LENGTH
)

test_dataset = LEVIRCCDataset(
    test_df,
    transform=transform,
    max_length=MAX_CAPTION_LENGTH
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 6815
Val: 1333
Test: 1929


In [7]:
# ============================================================
# BASELINE MODEL
# ============================================================

class ChangeCaptionBaseline(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_size=512
    ):

        super().__init__()

        # ----------------------------------------------------
        # Visual encoder
        # ----------------------------------------------------

        backbone = resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        self.encoder = nn.Sequential(
            *list(backbone.children())[:-1]
        )

        # Freeze initially
        for param in self.encoder.parameters():
            param.requires_grad = False

        self.feature_size = 512

        # ----------------------------------------------------
        # Combine A, B and their difference
        # ----------------------------------------------------

        self.visual_projection = nn.Linear(
            512 * 3,
            hidden_size
        )

        # ----------------------------------------------------
        # Caption embedding
        # ----------------------------------------------------

        self.embedding = nn.Embedding(
            vocab_size,
            hidden_size,
            padding_idx=PAD_IDX
        )

        # ----------------------------------------------------
        # LSTM decoder
        # ----------------------------------------------------

        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        # ----------------------------------------------------
        # Output vocabulary
        # ----------------------------------------------------

        self.output = nn.Linear(
            hidden_size,
            vocab_size
        )

    def encode_images(self, image_A, image_B):

        feature_A = self.encoder(
            image_A
        ).flatten(1)

        feature_B = self.encoder(
            image_B
        ).flatten(1)

        difference = torch.abs(
            feature_A - feature_B
        )

        visual = torch.cat(
            [
                feature_A,
                feature_B,
                difference
            ],
            dim=1
        )

        visual = self.visual_projection(
            visual
        )

        return visual

    def forward(
        self,
        image_A,
        image_B,
        captions
    ):

        visual = self.encode_images(
            image_A,
            image_B
        )

        # Caption input excludes final token
        caption_input = captions[:, :-1]

        embedded = self.embedding(
            caption_input
        )

        # Initial hidden state comes from images
        h0 = visual.unsqueeze(0)
        c0 = torch.zeros_like(h0)

        outputs, _ = self.decoder(
            embedded,
            (h0, c0)
        )

        logits = self.output(outputs)

        return logits


model = ChangeCaptionBaseline(
    vocab_size=VOCAB_SIZE
).to(DEVICE)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

1.4%

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Aishwarya Gilhotra/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


Trainable parameters: 3639517


In [8]:
# ============================================================
# TRAINING
# ============================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=LEARNING_RATE
)


def run_epoch(loader, training=True):

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0

    progress = tqdm(
        loader,
        leave=False
    )

    for image_A, image_B, captions in progress:

        image_A = image_A.to(
            DEVICE,
            non_blocking=True
        )

        image_B = image_B.to(
            DEVICE,
            non_blocking=True
        )

        captions = captions.to(
            DEVICE,
            non_blocking=True
        )

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):

            logits = model(
                image_A,
                image_B,
                captions
            )

            # Target excludes START token
            targets = captions[:, 1:]

            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                targets.reshape(-1)
            )

            if training:
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                optimizer.step()

        total_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return total_loss / len(loader)


history = []

for epoch in range(NUM_EPOCHS):

    train_loss = run_epoch(
        train_loader,
        training=True
    )

    val_loss = run_epoch(
        val_loader,
        training=False
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    print(
        f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
        f"Train: {train_loss:.4f} | "
        f"Val: {val_loss:.4f}"
    )

  0%|          | 0/213 [00:00<?, ?it/s]

  0%|          | 0/42 [00:00<?, ?it/s]

Epoch 01/10 | Train: 3.2105 | Val: 1.7936


  0%|          | 0/213 [00:00<?, ?it/s]

  0%|          | 0/42 [00:00<?, ?it/s]

...

  0%|          | 0/42 [00:00<?, ?it/s]

Epoch 09/10 | Train: 1.1228 | Val: 1.1474


  0%|          | 0/213 [00:00<?, ?it/s]

  0%|          | 0/42 [00:00<?, ?it/s]

Epoch 10/10 | Train: 1.0979 | Val: 1.1080


In [9]:
# ============================================================
# SAVE BASELINE
# ============================================================

MODEL_PATH = (
    CC_ROOT.parent
    / "change_captioning_baseline.pt"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "stoi": stoi,
        "itos": itos,
        "vocab_size": VOCAB_SIZE,
        "config": {
            "image_size": IMAGE_SIZE,
            "max_caption_length": MAX_CAPTION_LENGTH
        }
    },
    MODEL_PATH
)

print("Saved:", MODEL_PATH)

Saved: C:\College\Dissertation\Data\LEVIR-CC\change_captioning_baseline.pt


In [10]:
# ============================================================
# QUALITATIVE EVALUATION — GENERATE CAPTIONS
# ============================================================

def generate_caption(image_A, image_B, max_length=30):

    model.eval()

    image_A = image_A.unsqueeze(0).to(DEVICE)
    image_B = image_B.unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        visual = model.encode_images(
            image_A,
            image_B
        )

        h = visual.unsqueeze(0)
        c = torch.zeros_like(h)

        current_token = torch.tensor(
            [[START_IDX]],
            device=DEVICE
        )

        generated = []

        for _ in range(max_length):

            embedded = model.embedding(
                current_token
            )

            output, (h, c) = model.decoder(
                embedded,
                (h, c)
            )

            logits = model.output(output[:, -1, :])

            next_token = logits.argmax(
                dim=-1
            )

            token_id = next_token.item()

            if token_id == END_IDX:
                break

            if token_id != PAD_IDX:
                generated.append(
                    itos[token_id]
                )

            current_token = next_token.unsqueeze(0)

    return " ".join(generated)


# ------------------------------------------------------------
# Show predictions
# ------------------------------------------------------------

sample_indices = np.random.choice(
    len(val_dataset),
    size=10,
    replace=False
)

for idx in sample_indices:

    row = val_df.iloc[idx]

    image_A, image_B, _ = val_dataset[idx]

    prediction = generate_caption(
        image_A,
        image_B
    )

    print("=" * 80)
    print("IMAGE:", row["filename"])
    print("CHANGE FLAG:", row["changeflag"])

    print("\nPREDICTED:")
    print(prediction)

    print("\nGROUND TRUTH:")

    for i, caption in enumerate(
        row["captions"],
        start=1
    ):
        print(f"{i}. {caption}")

IMAGE: val_001003.png
CHANGE FLAG: 0

PREDICTED:
the scene is the same as before .

GROUND TRUTH:
1. there is no difference .
2. the two scenes seem identical .
3. the scene is the same as before .
4. no change has occurred .
5. almost nothing has changed .
IMAGE: val_001168.png
CHANGE FLAG: 0

PREDICTED:
the scene is the same as before .

GROUND TRUTH:
1. there is no difference .
2. the two scenes seem identical .
3. the scene is the same as before .
4. no change has occurred .
5. almost nothing has changed .
IMAGE: val_001357.png
CHANGE FLAG: 1

PREDICTED:
a road with a parking lot is built on the bareland .

GROUND TRUTH:
1. some vegetation disappears and roads and buildings appear .
2. a construction site appears at the bottom replacing the plants .
3. two buildings with some roads around are built replacing the forest .
4. a playground has been built and a house stands by it .
5. a giant building with roads appear among the plants .
IMAGE: val_000308.png
CHANGE FLAG: 1

PREDICTED:

In [16]:
# ============================================================
# FIXED BASELINE EVALUATION
# ============================================================

# ------------------------------------------------------------
# BLEU
# ------------------------------------------------------------

smooth = SmoothingFunction().method1

bleu_scores = {
    "BLEU-1": corpus_bleu(
        references,
        predictions,
        weights=(1, 0, 0, 0),
        smoothing_function=smooth
    ),

    "BLEU-2": corpus_bleu(
        references,
        predictions,
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=smooth
    ),

    "BLEU-3": corpus_bleu(
        references,
        predictions,
        weights=(1/3, 1/3, 1/3, 0),
        smoothing_function=smooth
    ),

    "BLEU-4": corpus_bleu(
        references,
        predictions,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smooth
    )
}


# ------------------------------------------------------------
# METEOR
# ------------------------------------------------------------

meteor_scores = []

for pred, refs in zip(predictions, references):

    # pred is already a list of tokens
    # refs is already a list of token lists

    score = meteor_score(
        refs,
        pred
    )

    meteor_scores.append(score)

meteor = np.mean(meteor_scores)


# ------------------------------------------------------------
# ROUGE-L
# ------------------------------------------------------------

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_scores = []

for pred, refs in zip(predictions, references):

    pred_text = " ".join(pred)

    sample_scores = []

    for ref in refs:

        ref_text = " ".join(ref)

        score = rouge.score(
            ref_text,
            pred_text
        )["rougeL"].fmeasure

        sample_scores.append(score)

    rouge_scores.append(
        max(sample_scores)
    )

rouge_l = np.mean(rouge_scores)


# ------------------------------------------------------------
# OVERALL RESULTS
# ------------------------------------------------------------

results = {
    **bleu_scores,
    "METEOR": meteor,
    "ROUGE-L": rouge_l
}

print("=" * 60)
print("BASELINE TEST RESULTS")
print("=" * 60)

for metric, score in results.items():
    print(f"{metric:10s}: {score:.4f}")


# ------------------------------------------------------------
# CHANGE / NO-CHANGE BREAKDOWN
# ------------------------------------------------------------

def evaluate_subset(mask):

    subset_predictions = [
        p
        for p, keep in zip(predictions, mask)
        if keep
    ]

    subset_references = [
        r
        for r, keep in zip(references, mask)
        if keep
    ]

    scores = {}

    # BLEU
    scores["BLEU-1"] = corpus_bleu(
        subset_references,
        subset_predictions,
        weights=(1, 0, 0, 0),
        smoothing_function=smooth
    )

    scores["BLEU-2"] = corpus_bleu(
        subset_references,
        subset_predictions,
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=smooth
    )

    scores["BLEU-3"] = corpus_bleu(
        subset_references,
        subset_predictions,
        weights=(1/3, 1/3, 1/3, 0),
        smoothing_function=smooth
    )

    scores["BLEU-4"] = corpus_bleu(
        subset_references,
        subset_predictions,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smooth
    )

    # METEOR
    meteor_values = [
        meteor_score(refs, pred)
        for pred, refs
        in zip(
            subset_predictions,
            subset_references
        )
    ]

    scores["METEOR"] = np.mean(
        meteor_values
    )

    # ROUGE-L
    rouge_values = []

    for pred, refs in zip(
        subset_predictions,
        subset_references
    ):

        pred_text = " ".join(pred)

        rouge_values.append(
            max(
                rouge.score(
                    " ".join(ref),
                    pred_text
                )["rougeL"].fmeasure
                for ref in refs
            )
        )

    scores["ROUGE-L"] = np.mean(
        rouge_values
    )

    return scores


no_change_mask = (
    evaluation_df["changeflag"].values == 0
)

change_mask = (
    evaluation_df["changeflag"].values == 1
)

no_change_scores = evaluate_subset(
    no_change_mask
)

change_scores = evaluate_subset(
    change_mask
)

breakdown = pd.DataFrame({
    "No Change": no_change_scores,
    "Change": change_scores
})

print("\n" + "=" * 60)
print("BASELINE — CHANGE VS NO-CHANGE")
print("=" * 60)

display(
    breakdown.round(4)
)


# ------------------------------------------------------------
# SAVE RESULTS
# ------------------------------------------------------------

evaluation_df.to_csv(
    "baseline_test_predictions.csv",
    index=False
)

pd.DataFrame([results]).to_csv(
    "baseline_metrics.csv",
    index=False
)

print("\nResults saved.")

BASELINE TEST RESULTS
BLEU-1    : 0.7438
BLEU-2    : 0.6034
BLEU-3    : 0.5162
BLEU-4    : 0.4563
METEOR    : 0.6561
ROUGE-L   : 0.6386


NameError: name 'evaluation_df' is not defined

In [17]:
# ============================================================
# CHANGE vs NO-CHANGE BASELINE EVALUATION
# ============================================================

# Reconstruct evaluation metadata from test_df
evaluation_df = test_df.iloc[:len(predictions)].copy().reset_index(drop=True)

evaluation_df["prediction"] = [
    " ".join(p) for p in predictions
]

# Make sure references align
evaluation_df["references"] = [
    [" ".join(r) for r in refs]
    for refs in references
]

print("Evaluation samples:", len(evaluation_df))
print(
    evaluation_df["changeflag"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# Evaluate one subset
# ------------------------------------------------------------

def evaluate_subset(mask):

    subset_predictions = [
        p
        for p, keep in zip(predictions, mask)
        if keep
    ]

    subset_references = [
        r
        for r, keep in zip(references, mask)
        if keep
    ]

    scores = {}

    # BLEU-1
    scores["BLEU-1"] = corpus_bleu(
        subset_references,
        subset_predictions,
        weights=(1, 0, 0, 0),
        smoothing_function=smooth
    )

    # BLEU-2
    scores["BLEU-2"] = corpus_bleu(
        subset_references,
        subset_predictions,
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=smooth
    )

    # BLEU-3
    scores["BLEU-3"] = corpus_bleu(
        subset_references,
        subset_predictions,
        weights=(1/3, 1/3, 1/3, 0),
        smoothing_function=smooth
    )

    # BLEU-4
    scores["BLEU-4"] = corpus_bleu(
        subset_references,
        subset_predictions,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smooth
    )

    # METEOR
    meteor_values = [
        meteor_score(refs, pred)
        for pred, refs
        in zip(
            subset_predictions,
            subset_references
        )
    ]

    scores["METEOR"] = np.mean(
        meteor_values
    )

    # ROUGE-L
    rouge_values = []

    for pred, refs in zip(
        subset_predictions,
        subset_references
    ):

        pred_text = " ".join(pred)

        sample_scores = [
            rouge.score(
                " ".join(ref),
                pred_text
            )["rougeL"].fmeasure
            for ref in refs
        ]

        rouge_values.append(
            max(sample_scores)
        )

    scores["ROUGE-L"] = np.mean(
        rouge_values
    )

    return scores


# ------------------------------------------------------------
# CHANGE / NO-CHANGE
# ------------------------------------------------------------

no_change_mask = (
    evaluation_df["changeflag"].values == 0
)

change_mask = (
    evaluation_df["changeflag"].values == 1
)

no_change_scores = evaluate_subset(
    no_change_mask
)

change_scores = evaluate_subset(
    change_mask
)


breakdown = pd.DataFrame({
    "No Change": no_change_scores,
    "Change": change_scores
})


print("\n" + "=" * 70)
print("BASELINE — CHANGE VS NO-CHANGE")
print("=" * 70)

display(
    breakdown.round(4)
)


# ------------------------------------------------------------
# Save complete baseline predictions
# ------------------------------------------------------------

evaluation_df.to_csv(
    "baseline_test_predictions.csv",
    index=False
)

pd.DataFrame([results]).to_csv(
    "baseline_metrics.csv",
    index=False
)

print("\nSaved:")
print("  baseline_test_predictions.csv")
print("  baseline_metrics.csv")

Evaluation samples: 1929
changeflag
0    965
1    964
Name: count, dtype: int64

BASELINE — CHANGE VS NO-CHANGE


,No Change,Change
BLEU-1,0.8582,0.6663
BLEU-2,0.8263,0.4409
BLEU-3,0.8138,0.2923
BLEU-4,0.8054,0.1942
METEOR,0.8653,0.4468
ROUGE-L,0.8633,0.4137



Saved:
  baseline_test_predictions.csv
  baseline_metrics.csv


In [18]:
# ============================================================
# QUICK DIAGNOSIS: 206 MISSING TEST SAMPLES
# ============================================================

# 1. All actual CC test images
cc_test_dir = CC_ROOT / "images" / "test" / "A"

all_cc_test = sorted(
    p.name for p in cc_test_dir.glob("*.png")
)

# 2. Images currently present in our evaluation
evaluated = set(evaluation_df["filename"])

# 3. Find missing CC test images
missing = [
    f for f in all_cc_test
    if f not in evaluated
]

print("Actual CC test images :", len(all_cc_test))
print("Evaluated samples     :", len(evaluated))
print("Missing               :", len(missing))

print("\nFirst 30 missing:")
print(missing[:30])


# ------------------------------------------------------------
# 4. Check whether captions exist for missing images
# ------------------------------------------------------------

caption_filenames = set(
    record["filename"]
    for record in images_data
)

missing_with_captions = [
    f for f in missing
    if f in caption_filenames
]

missing_without_captions = [
    f for f in missing
    if f not in caption_filenames
]

print("\nMissing WITH captions :", len(missing_with_captions))
print("Missing WITHOUT captions:", len(missing_without_captions))


# ------------------------------------------------------------
# 5. Check whether they have predictions
# ------------------------------------------------------------

prediction_count = len(predictions)

print("\nPredictions generated :", prediction_count)

# Show a few examples
print("\nMissing examples:")
for f in missing[:10]:
    print(f)

Actual CC test images : 2135
Evaluated samples     : 1929
Missing               : 206

First 30 missing:
['test_000027.png', 'test_000028.png', 'test_000030.png', 'test_000037.png', 'test_000038.png', 'test_000039.png', 'test_000040.png', 'test_000075.png', 'test_000097.png', 'test_000098.png', 'test_000201.png', 'test_000223.png', 'test_000244.png', 'test_000263.png', 'test_000273.png', 'test_000290.png', 'test_000359.png', 'test_000503.png', 'test_000507.png', 'test_000547.png', 'test_000568.png', 'test_000625.png', 'test_000670.png', 'test_000732.png', 'test_000736.png', 'test_000750.png', 'test_000751.png', 'test_000762.png', 'test_000766.png', 'test_000769.png']


NameError: name 'images_data' is not defined

In [19]:
# ============================================================
# QUICK CHECK: ARE THE 206 MISSING IMAGES IN THE CC JSON?
# ============================================================

import json

caption_json = CC_ROOT / "LevirCCcaptions.json"

with open(caption_json, "r", encoding="utf-8") as f:
    caption_data = json.load(f)

caption_records = caption_data["images"]

caption_filenames = {
    record["filename"]
    for record in caption_records
}

missing_with_captions = [
    f for f in missing
    if f in caption_filenames
]

missing_without_captions = [
    f for f in missing
    if f not in caption_filenames
]

print("Missing CC test images :", len(missing))
print("Missing WITH captions  :", len(missing_with_captions))
print("Missing WITHOUT captions:", len(missing_without_captions))

print("\nFirst 20 WITH captions:")
print(missing_with_captions[:20])

print("\nFirst 20 WITHOUT captions:")
print(missing_without_captions[:20])

Missing CC test images : 206
Missing WITH captions  : 0
Missing WITHOUT captions: 206

First 20 WITH captions:
[]

First 20 WITHOUT captions:
['test_000027.png', 'test_000028.png', 'test_000030.png', 'test_000037.png', 'test_000038.png', 'test_000039.png', 'test_000040.png', 'test_000075.png', 'test_000097.png', 'test_000098.png', 'test_000201.png', 'test_000223.png', 'test_000244.png', 'test_000263.png', 'test_000273.png', 'test_000290.png', 'test_000359.png', 'test_000503.png', 'test_000507.png', 'test_000547.png']
